# CAMP 1: Medical Image Registration

## In-person practical

Work in pairs, one step at a time: inspect the input, choose settings, predict, run, then explain the result **before going to the next step**. Change one setting at a time. Running the whole notebook at once skips the investigation.

The intensity experiments use real CT and MR slices from the previous course material. A known displacement is added for controlled experiments. The point and regularization experiments remain synthetic so we can evaluate their geometry independently. No GPU or downloads are needed after setup.

Each settings cell is followed by a result cell. Logs keep your earlier trials. Rerunning an input/setup cell resets that section's log. The two short coding tasks have working reference fallbacks. The student notebook opens without saved answers.

In [ ]:
# Install missing registration dependencies; on Colab upload the student ZIP once.
import sys
from pathlib import Path
try:
    from google.colab import files as colab_files
    on_colab = True
except ImportError:
    on_colab = False
if not on_colab:
    import importlib.util
    import subprocess
    if importlib.util.find_spec("torch") is None:
        print("Installing PyTorch into the notebook's Python environment...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "torch>=2.2,<3"])
        importlib.invalidate_caches()
# SimpleITK is needed locally and on Colab for Mattes mutual information.
import importlib.util
import subprocess
if importlib.util.find_spec("SimpleITK") is None:
    print("Installing SimpleITK for Mattes mutual information...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "SimpleITK>=2.5,<3"])
    importlib.invalidate_caches()
if on_colab and not (Path.cwd()/"registration_practical"/"registration_lab.py").exists() and not (Path.cwd()/"registration_lab.py").exists():
    import io, zipfile
    print("Upload CAMP1_registration_practical_student.zip (includes helper and CT crops).")
    uploaded = colab_files.upload()
    archive_name = next(name for name in uploaded if name.endswith(".zip"))
    with zipfile.ZipFile(io.BytesIO(uploaded[archive_name])) as archive:
        root = Path.cwd().resolve()
        for name in archive.namelist():
            if not (root/name).resolve().is_relative_to(root):
                raise ValueError("Unexpected path in archive.")
        archive.extractall(root)
    del uploaded


## 0. Setup and coordinate conventions

- Points use `(x, y)`; image arrays use `[y, x]`.
- ICP estimates a **moving-to-fixed** point map `T(q) = Rq + t`.
- Images use a **fixed-to-moving** sampling map `phi(p)`: the output at `p` samples `moving(phi(p))`.
- Image parameters are `(angle in degrees, tx in pixels, ty in pixels)`, rotating about the image center. With y pointing down, positive angles appear clockwise.
- The CT/MR slice grid has 2 mm pixels. The simplified 2D optimizer works in pixel coordinates; point and deformation examples have unit pixel spacing.

Opening discussion: does a decreasing registration loss establish better anatomical alignment? Which direction must the sampling map point?

In [ ]:
from pathlib import Path
import sys
import importlib
import numpy as np
import scipy
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
%matplotlib inline

candidates = [Path.cwd(), Path.cwd().parent, Path.cwd()/"registration_practical",
              Path.cwd()/"new camp exercise"/"registration_practical"]
LAB_DIR = next((p for p in candidates if (p/"registration_lab.py").is_file()), None)
if LAB_DIR is None:
    raise FileNotFoundError("Keep the registration_practical folder intact and open this notebook there.")
sys.path.insert(0, str(LAB_DIR))
import registration_lab as lab
lab = importlib.reload(lab)  # Pick up helper updates when this setup cell is rerun.
if not hasattr(lab, "medical_intensity_case"):
    raise ImportError(f"Outdated helper at {lab.__file__}. Use the complete updated registration_practical folder.")

def show(fig):
    display(fig)
    plt.close(fig)

def show_trial_log(rows, columns):
    def text(value):
        if isinstance(value, (float, np.floating)):
            return f"{value:.4g}"
        return str(value).replace("|", "/").replace("\n", " ")
    lines = ["| " + " | ".join(label for key, label in columns) + " |",
             "| " + " | ".join("---" for _ in columns) + " |"]
    lines += ["| " + " | ".join(text(row[key]) for key, label in columns) + " |" for row in rows]
    display(Markdown("\n".join(lines)))

image_columns = [("trial", "Trial"), ("model", "Model"), ("metric", "Metric"),
                 ("initial", "Initial pose"), ("final", "Final pose"), ("score", "Metric score"),
                 ("error", "Imposed-motion error [px]"), ("overlap", "Compared pixels"), ("prediction", "Prediction")]
def run_image_trial(case, log, model, metric, initial_pose, prediction):
    result = lab.register_intensity(case, metric, objective_direction, initial_pose, model=model)
    log.append(dict(trial=len(log)+1, model=model, metric=metric, initial=tuple(initial_pose),
                    final=tuple(np.round(result["parameters"], 3)), score=result["score"],
                    error=result["landmark_error"], overlap=result["overlap_pixels"], prediction=prediction or "Not recorded"))
    show(lab.plot_intensity(case, {f"Trial {len(log)}: {model}, {metric}": result}))
    print("Stop:", result["message"])
    print("Imposed-motion error:", round(result["landmark_error"], 3), "pixels =",
          round(2*result["landmark_error"], 3), "mm on this grid")
    return result

print("Python", sys.version.split()[0], "| NumPy", np.__version__, "| SciPy", scipy.__version__)
print("Intensity registration: source units, full valid image overlap; no circular ROI.")
print("Setup ready. Run one step, inspect it, then continue.")

## 1. ICP: correspondences and initialization

### 1.1 Inspect the point sets

The moving contour is a rotated, translated and shuffled copy. Original point labels are withheld from ICP and used only for evaluation. Run this setup once. Identify a plausible alignment from the geometry.

In [ ]:
point_data = lab.point_case()
icp_trials = []
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(*point_data["fixed"].T, label="Fixed", s=14)
ax.scatter(*point_data["moving"].T, label="Moving", s=14)
ax.set(xlabel="x [pixels]", ylabel="y [pixels]", aspect="equal")
ax.legend()
show(fig)

### 1.2 Choose correspondences

Replace the fallback with the nearest fixed-point index for each moving point. The pairwise squared-distance matrix has shape `(number of moving points, number of fixed points)`. Which axis should you minimize? The rigid solver is supplied.

In [ ]:
def correspondences(current_moving, fixed):
    indices = None  # TODO: return the nearest fixed-point index for each moving point.
    return lab.nearest_indices(current_moving, fixed) if indices is None else indices

### 1.3 Choose an initial pose, then inspect its result

Start from the identity below or choose your own pose. Write a prediction before running. The result shows initial/final correspondence lines and two error curves.

In [ ]:
initial_angle = 0.0
initial_translation = (0.0, 0.0)
max_iterations = 60
prediction = ""

In [ ]:
current_icp = lab.run_icp(point_data, initial_angle, initial_translation, correspondences,
                          max_iterations=max_iterations)
icp_trials.append(dict(trial=len(icp_trials)+1, angle=initial_angle, translation=tuple(initial_translation),
                       limit=max_iterations, nn=float(current_icp["nn_error"][-1]),
                       true=float(current_icp["true_error"][-1]), prediction=prediction or "Not recorded"))
show(lab.plot_icp(point_data, {f"Trial {len(icp_trials)}": current_icp}))
show_trial_log(icp_trials, [("trial","Trial"),("angle","Initial angle"),("translation","Initial translation"),
    ("limit","Iteration limit"),("nn","Nearest-neighbor RMSE [px]"),
    ("true","Known-correspondence RMSE [px]"),("prediction","Prediction")])

### 1.4 Change initialization

Choose a substantially different angle. Keep translation and iteration limit fixed. Run this trial and compare it with your first result. Find two starts with different outcomes; rerun this step as needed.

In [ ]:
initial_angle = 170.0  # Choose a different angle to investigate.
prediction = ""

In [ ]:
current_icp = lab.run_icp(point_data, initial_angle, initial_translation, correspondences,
                          max_iterations=max_iterations)
icp_trials.append(dict(trial=len(icp_trials)+1, angle=initial_angle, translation=tuple(initial_translation),
                       limit=max_iterations, nn=float(current_icp["nn_error"][-1]),
                       true=float(current_icp["true_error"][-1]), prediction=prediction or "Not recorded"))
show(lab.plot_icp(point_data, {f"Trial {len(icp_trials)}": current_icp}))
show_trial_log(icp_trials, [("trial","Trial"),("angle","Initial angle"),("translation","Initial translation"),
    ("limit","Iteration limit"),("nn","Nearest-neighbor RMSE [px]"),
    ("true","Known-correspondence RMSE [px]"),("prediction","Prediction")])

### 1.5 Change only the iteration budget

Keep the current initial pose. Predict whether extra iterations can fix its outcome, then change the limit. This reruns from the same initialization; it does not continue from the previous final pose.

In [ ]:
max_iterations = 120
prediction = ""

In [ ]:
current_icp = lab.run_icp(point_data, initial_angle, initial_translation, correspondences,
                          max_iterations=max_iterations)
icp_trials.append(dict(trial=len(icp_trials)+1, angle=initial_angle, translation=tuple(initial_translation),
                       limit=max_iterations, nn=float(current_icp["nn_error"][-1]),
                       true=float(current_icp["true_error"][-1]), prediction=prediction or "Not recorded"))
show(lab.plot_icp(point_data, {f"Trial {len(icp_trials)}": current_icp}))
show_trial_log(icp_trials, [("trial","Trial"),("angle","Initial angle"),("translation","Initial translation"),
    ("limit","Iteration limit"),("nn","Nearest-neighbor RMSE [px]"),
    ("true","Known-correspondence RMSE [px]"),("prediction","Prediction")])

## 2. CT–CT: sampling and transformation model

### 2.1 Inspect the medical image pair

The fixed image is an abdominal CT slice from the supplied volume. The moving image is that same slice with an imposed rigid displacement. Registration uses the original CT intensity units with mild Gaussian smoothing (0.65 pixels). No clipping or intensity rescaling is applied to the registration inputs. The [-150, 250] window is used only to display CT. This is real anatomy with controlled motion.

All three metrics use the full valid image overlap, including real dark background pixels. There is no circular ROI or anatomical mask. Only out-of-image samples and padding introduced by generating motion are excluded. Search bounds are angle [-20, 20] degrees and translations [-10, 10] pixels. The overlap changes with the transform; the trial log reports the number of compared pixels. Cyan/red overlays are windowed for display only.

Run this cell once; it starts the CT trial log.

In [ ]:
ct_case = lab.medical_intensity_case("ct_ct")
ct_trials = []
show(lab.plot_image_inputs(ct_case))

### 2.2 Move the image manually

Change one component at a time, then rerun the display. These are **sampling-map** parameters, so reason about the direction in which structures appear to move. This cell does no optimization.

In [ ]:
manual_pose = (0.0, 5.0, 0.0)  # angle, tx, ty
show(lab.plot_manual_pose(ct_case, manual_pose))

### 2.3 Define the objective direction

Choose +1 or -1 for each metric so a minimizer follows the correct direction. `None` uses the supplied fallback.

SSD is the sum of squared intensity differences. NCC is **signed**, mean-centered correlation. MI uses SimpleITK’s Mattes estimator with 32 histogram bins and Parzen-window smoothing. Scores are positive mutual information in bits; SimpleITK’s negative-MI value is converted to this convention. The same Powell optimizer, bounds and valid-overlap rule are used for all three metrics. SSD remains a sum of squared differences; it is not divided by the overlap size. The three raw scores have different scales; do not compare them numerically as if they measured the same quantity.

In [ ]:
def objective_direction(metric):
    choices = {"SSD": None, "NCC": None, "MI": None}
    return lab.metric_sign(metric) if choices[metric] is None else choices[metric]

### 2.4 Fit a translation-only model

Keep SSD and the identity start for this baseline. Predict which misalignment translation can remove.

The displayed **imposed-motion error** is mean displacement error at five fixed virtual probe locations, withheld from the optimizer. These are not manually annotated landmarks. In this CT–CT construction, the imposed transform is known ground truth, up to interpolation effects.

In [ ]:
model = "translation"
metric = "SSD"
initial_pose = (0.0, 0.0, 0.0)
prediction = ""

Run the next cell to inspect **only these settings**. It appends this trial to the log. Rerun the settings and result cells to try again; keep the other choices fixed.

In [ ]:
ct_translation = run_image_trial(ct_case, ct_trials, model, metric, initial_pose, prediction)
show_trial_log(ct_trials, image_columns)

### 2.5 Add rotation: rigid registration

Change only the transformation model. Keep the metric, initialization and optimizer unchanged. Inspect the result before continuing.

In [ ]:
model = "rigid"
metric = "SSD"
initial_pose = (0.0, 0.0, 0.0)
prediction = ""

Run the next cell to inspect **only these settings**. It appends this trial to the log. Rerun the settings and result cells to try again; keep the other choices fixed.

In [ ]:
ct_rigid = run_image_trial(ct_case, ct_trials, model, metric, initial_pose, prediction)
show_trial_log(ct_trials, image_columns)

### 2.6 Test your initialization

Choose a new start within the bounds above. Keep the rigid model and SSD fixed. Use the previous images to justify your prediction; a different start need not produce a worse result.

In [ ]:
model = "rigid"
metric = "SSD"
initial_pose = (-10.0, -5.0, 5.0)  # Choose your own starting pose.
prediction = ""

Run the next cell to inspect **only these settings**. It appends this trial to the log. Rerun the settings and result cells to try again; keep the other choices fixed.

In [ ]:
ct_current = run_image_trial(ct_case, ct_trials, model, metric, initial_pose, prediction)
show_trial_log(ct_trials, image_columns)

### 2.7 Inspect the objective around your result

This sweep changes horizontal translation only; angle and vertical translation stay at **your last result**. It is a slice through the objective, not the complete landscape or optimizer trajectory.

In [ ]:
show(lab.plot_selected_landscape(ct_case, ct_current, ct_current["metric"], objective_direction))

## 3. CT–MR: choose a similarity measure

### 3.1 Inspect the modalities before optimizing

These slices come from the supplied CT and MR volumes, at the same physical plane on verified identical grids. CT and MR retain their original intensity units, with Gaussian smoothing of 0.65 pixels. No intensity rescaling, clipping or histogram matching is applied before SSD, NCC or MI. CT windowing and the MR display range affect plots only. The MR slice then receives the same imposed rigid displacement used above. No fitted alignment is applied during preparation.

**Evaluation limitation:** the supplied CT–MR pair has no independently validated anatomical correspondence. Its imposed-motion error measures return toward the supplied reference placement; it is **not anatomical registration error**. A shared physical grid does not guarantee aligned organs. Breathing and out-of-plane differences can remain.

SSD assumes corresponding structures have comparable intensities. CT and MR do not share an intensity scale or a consistent tissue-to-intensity relationship. A plausible SSD result on one pair does not establish suitability for multimodal registration.

In [ ]:
mr_case = lab.medical_intensity_case("ct_mr")
mr_trials = []
show(lab.plot_image_inputs(mr_case))

### 3.2 Establish one metric baseline

Run SSD first on the source intensities using a rigid model and the identity start. Then compare Mattes MI with the same model and initialization. Inspect anatomical boundaries as well as the limited displacement-recovery diagnostic.

In [ ]:
model = "rigid"
metric = "SSD"  # Choose one metric for your baseline.
initial_pose = (0.0, 0.0, 0.0)
prediction = ""

Run the next cell to inspect **only these settings**. It appends this trial to the log. Rerun the settings and result cells to try again; keep the other choices fixed.

In [ ]:
mr_baseline = run_image_trial(mr_case, mr_trials, model, metric, initial_pose, prediction)
show_trial_log(mr_trials, image_columns)

### 3.3 Change only the similarity measure

Choose a different metric. Preserve the model and initialization used in step 3.2 for a controlled comparison. Try a third metric by rerunning this step if useful. With source intensities and the full valid overlap, the default SSD run misaligns this pair while Mattes MI recovers the added motion more closely. Inspect anatomical alignment as well as the limited imposed-motion diagnostic.

In [ ]:
metric = "MI"  # Choose a different metric; "NCC" is also available.
prediction = ""

Run the next cell to inspect **only these settings**. It appends this trial to the log. Rerun the settings and result cells to try again; keep the other choices fixed.

In [ ]:
mr_current = run_image_trial(mr_case, mr_trials, model, metric, initial_pose, prediction)
show_trial_log(mr_trials, image_columns)

### 3.4 Inspect your chosen metric, then decide

Inspect a translation sweep for the metric from step 3.3. If needed, return to that step and change one setting. State which result you would investigate further and what evidence is missing.

In [ ]:
show(lab.plot_selected_landscape(mr_case, mr_current, mr_current["metric"], objective_direction))

## 4. Deformation: flexibility and regularization

### 4.1 Inspect the pair and run a rigid baseline

Each result displays the target, moving and registered images, before/after overlays, a sampling grid, differences to the target, and the Jacobian. The two difference maps share one signed scale (image minus target), fixed from the unregistered pair. Gray marks excluded or unavailable samples.

This controlled synthetic pair contains smooth local motion, noise and a local appearance change. It isolates regularization while providing a known field and held-out landmarks. The medical intensity experiments above do not supply such deformation ground truth.

The next models minimize `E(u) = S(u) + lambda * R(u)`, with `phi(p) = p + u(p)`. `S` is mean squared image mismatch in a fixed interior region. `R` is mean squared first differences over both axes and components, with unit pixel spacing.

In [ ]:
deformation = lab.deformable_case()
deformation_trials = []
show(lab.plot_fields(deformation, {"Before registration": np.zeros_like(deformation["truth"])}))

In [ ]:
rigid_initial = lab.rigid_for_deformation(deformation)
show(lab.plot_fields(deformation, {"Rigid baseline": rigid_initial}))
lab.print_field_results(deformation, {"Rigid baseline": rigid_initial})

### 4.2 Allow local motion without regularization

The dense field has two displacement components per pixel, each bounded to [-6, 6]. Run one unregularized trial from the rigid baseline. Inspect the warped grid, held-out landmark error and Jacobian map before adding a penalty.

In [ ]:
live_lambda = 0.0
prediction = ""

In [ ]:
live_result = lab.optimize_field(deformation, weight=live_lambda, initial=rigid_initial,
                                 max_iterations=250)
deformation_trials.append(dict(trial=len(deformation_trials)+1, weight=live_lambda,
    mse=live_result["mse"], regularity=live_result["regularity"], error=live_result["landmark_error"],
    folds=live_result["fold_percent"], prediction=prediction or "Not recorded"))
show(lab.plot_fields(deformation, {f"lambda = {live_lambda}": live_result["field"]}))
print("Iterations:", live_result["iterations"], "| Stop:", live_result["message"])
show_trial_log(deformation_trials, [("trial","Trial"),("weight","Lambda"),("mse","Image MSE"),
    ("regularity","Regularity penalty"),("error","Landmark error [px]"),
    ("folds","Nonpositive Jacobians [%]"),("prediction","Prediction")])
fig, ax = plt.subplots(figsize=(6, 2.8))
ax.plot(live_result["history"])
ax.set(xlabel="Optimizer iteration", ylabel="S + lambda R")
show(fig)

### 4.3 Add regularization, then choose another weight

Choose a positive weight. Keep the representation, displacement bounds, rigid initialization and 250-iteration cap fixed. Inspect this result, then rerun with a substantially larger weight. The log retains both.

The Jacobian is computed from `identity + u`, not from `u` alone. Positive sampled determinants do not prove global invertibility. Compare image and geometric measures separately; total E changes its meaning when lambda changes. Reaching the iteration cap is not a claim of convergence.

In [ ]:
live_lambda = 0.01  # Choose a positive weight, then investigate a larger one.
prediction = ""

In [ ]:
live_result = lab.optimize_field(deformation, weight=live_lambda, initial=rigid_initial,
                                 max_iterations=250)
deformation_trials.append(dict(trial=len(deformation_trials)+1, weight=live_lambda,
    mse=live_result["mse"], regularity=live_result["regularity"], error=live_result["landmark_error"],
    folds=live_result["fold_percent"], prediction=prediction or "Not recorded"))
show(lab.plot_fields(deformation, {f"lambda = {live_lambda}": live_result["field"]}))
print("Iterations:", live_result["iterations"], "| Stop:", live_result["message"])
show_trial_log(deformation_trials, [("trial","Trial"),("weight","Lambda"),("mse","Image MSE"),
    ("regularity","Regularity penalty"),("error","Landmark error [px]"),
    ("folds","Nonpositive Jacobians [%]"),("prediction","Prediction")])
fig, ax = plt.subplots(figsize=(6, 2.8))
ax.plot(live_result["history"])
ax.set(xlabel="Optimizer iteration", ylabel="S + lambda R")
show(fig)

## 5. Learning registration from image agreement

The question for this section is: **how does comparing a warped image with a fixed image teach a network to predict a displacement field?**

A small encoder/decoder receives two CT images and predicts two offset components on a **5×5 control grid**. Bicubic interpolation expands this coarse prediction to a dense sampling field, followed by smooth bounding to ±12 pixels. This prevents the model from fitting independent pixel-scale ripples; it is not a cubic B-spline or a biomechanical model. A differentiable sampler warps the moving image. Image mismatch and a smoothness penalty train the shared network weights. This illustrates the VoxelMorph principle with a smaller teaching architecture, not a reproduction of the published model.

The images are real CT crops at their native 128×128 resolution, with no downsampling. Broad regional motion is generated from a **3×3 control grid** plus translation, with smooth bounding and known backward displacements. The amplitude is 12 pixels per component before smooth bounding (2.4 times the earlier example). Control nodes span the image, so the motion varies over large regions instead of following individual image edges. The known field is used only for evaluation, never for training. Training and evaluation source slices are disjoint but come from one volume; this is not an evaluation on independent patients. Both splits use the same motion generator. Distances here are source-grid pixels (2 mm per pixel).

Run each step and inspect it before continuing. The network is initialized from scratch and trained on CPU in this notebook. For local setup, install the requirements and run the first setup cell; it automatically installs PyTorch into the notebook's Python environment if missing. In Colab, use the default CPU runtime and upload the student ZIP in the setup cell above. No Google Drive mount is needed.

### 5.1 Run the network before training

Run this setup once. It resets the learning model and log and creates a new image pair. After updating the helper files, rerun this step: reloading a module alone does not replace an existing image pair or network. Check the printed version and displacement below. The last layer starts near zero, so the initial displacement is close to identity. Inspect the target, moving and registered images, both overlays, the sampling grid and Jacobian. The before/after difference maps show image minus target with the same signed color scale, anchored to the unregistered pair. Gray areas are outside the evaluation ROI or unavailable. Compare the initial overlay with the network output.

In [ ]:
from pathlib import Path
import sys, importlib, copy
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
%matplotlib inline
try:
    import torch
except ImportError as exc:
    raise ImportError('Install PyTorch with %pip install "torch>=2.2,<3", then rerun this cell.') from exc

candidates = [Path.cwd(), Path.cwd().parent, Path.cwd()/"registration_practical",
              Path.cwd()/"new camp exercise"/"registration_practical"]
DL_DIR = next((p for p in candidates if (p/"registration_learning.py").is_file()), None)
if DL_DIR is None:
    raise FileNotFoundError("Keep the complete updated practical folder together; in Colab upload its ZIP first.")
sys.path.insert(0, str(DL_DIR))
import registration_lab as lab
lab = importlib.reload(lab)
import registration_learning as dl
dl = importlib.reload(dl)
expected_version = "coarse5x5_motion12_v2"
if getattr(dl, "ARCHITECTURE_VERSION", None) != expected_version:
    raise RuntimeError(f"Outdated helper at {dl.__file__}. Replace the complete practical folder with the updated student ZIP; on Colab extract the new ZIP or start a fresh runtime.")
torch.set_num_threads(2)

def show_learning(fig):
    display(fig)
    plt.close(fig)

def learning_log(rows):
    lines = ["| Stage | Updates | Image MSE | R | Field error [px] | Nonpositive Jacobians [%] |",
             "| --- | --- | --- | --- | --- | --- |"]
    for name, updates, result in rows:
        lines.append(f"| {name} | {updates} | {result['image']:.6f} | {result['regularity']:.5f} | {result['field_error']:.3f} | {result['folds']:.2f} |")
    display(Markdown("\n".join(lines)))

network = dl.new_model()  # Fresh weights; no checkpoint is loaded.
optimizer = torch.optim.Adam(network.parameters(), lr=dl.LEARNING_RATE)
learning_updates = 0
learning_history = []
learning_trials = []
training_example = dl.make_batch("train", seed=42, index=6)
untrained_result = dl.evaluate(network, training_example)
learning_trials.append(("Before training", learning_updates, untrained_result))
motion = training_example["truth"].norm(dim=1)[:, dl.MARGIN:-dl.MARGIN, dl.MARGIN:-dl.MARGIN]
print("Learning version:", dl.ARCHITECTURE_VERSION)
print("Helper:", Path(dl.__file__).resolve())
print(f"Generated displacement in loss ROI: mean {motion.mean().item():.2f} px; maximum {motion.max().item():.2f} px")
print("Device: CPU | Parameters:", sum(p.numel() for p in network.parameters()), "| PyTorch:", torch.__version__)
show_learning(dl.plot_result(training_example, untrained_result, "Before training"))
learning_log(learning_trials)

### 5.2 Compute the training loss

The objective is `E = S + lambda * R`. `S` is image MSE in an interior region. `R` averages squared neighboring differences of the predicted displacement. The weight is 0.01 in the training loop below. The coarse output representation also limits spatial detail; this is separate from the regularization penalty. The field is bounded to ±12 pixels; a sixteen-pixel margin excludes padding from the image loss.

Read the arguments to the loss below: no ground-truth field is passed. Compare the two contributions. Is a small smoothness penalty by itself evidence of a useful registration?

In [ ]:
network.eval()
with torch.no_grad():
    predicted_field = network(training_example["fixed"], training_example["moving"])
    warped_image = dl.warp(training_example["moving"], predicted_field)
    total_loss, image_loss, regularity = dl.losses(
        training_example["fixed"], training_example["moving"], predicted_field)
print(f"Image mismatch S = {image_loss.item():.6f}")
print(f"Regularity R = {regularity.item():.6f}; weighted contribution = {dl.WEIGHT*regularity.item():.6f}")
show_learning(dl.plot_loss_parts(dl.evaluate(network, training_example)))

### 5.3 Follow one training update

Predict what changes: the input images, the field values, or the shared network weights? Run the update below. It shows the image-loss gradient with respect to displacement and checks that network weights changed. The images and known evaluation field stay fixed. Rerunning this cell performs another update on this same pair; the following training block uses many generated pairs.

In [ ]:
network.train()
optimizer.zero_grad(set_to_none=True)
weights_before = torch.cat([p.detach().flatten().clone() for p in network.parameters()])
predicted_field = network(training_example["fixed"], training_example["moving"])
total_loss, image_loss, regularity = dl.losses(
    training_example["fixed"], training_example["moving"], predicted_field)
# Inspect the learning signal passed through the sampler, before updating weights.
image_gradient = torch.autograd.grad(image_loss, predicted_field, retain_graph=True)[0]
total_loss.backward()
network_gradient = torch.sqrt(sum(p.grad.square().sum() for p in network.parameters() if p.grad is not None))
optimizer.step()
learning_updates += 1
weights_after = torch.cat([p.detach().flatten() for p in network.parameters()])
print("Network gradient norm:", network_gradient.item())
print("Weight-change norm:", torch.linalg.vector_norm(weights_after-weights_before).item())
show_learning(dl.plot_gradient(training_example, {"field_gradient": image_gradient[0].norm(dim=0).detach().numpy()}))
one_update_result = dl.evaluate(network, training_example)
learning_trials.append(("Live weight update", learning_updates, one_update_result))
show_learning(dl.plot_result(training_example, one_update_result, "After one weight update"))
learning_log(learning_trials)

### 5.4 Train for 1,000 additional updates

Continue training the network from the single update above. This cell performs **1,000 additional optimizer updates**, using batches of eight newly generated training pairs. Starting at step 5.1 and running step 5.3 once gives **1,001 updates in total**. The same optimizer and weights continue throughout; no saved model is loaded.

Write a prediction, then run the training cell. Progress and batch loss are printed every 100 updates. Compare the trained output with the earlier stages on the same displayed example. Rerunning the training cell adds another 1,000 updates; rerun step 5.1 to start over.

In [ ]:
prediction = ""  # Predict how the alignment and field will change.

In [ ]:
training_steps = 1000
for block_step in range(1, training_steps + 1):
    batch = dl.make_batch("train", count=8, seed=1000 + learning_updates)
    step_result = dl.train_update(network, optimizer, batch)
    learning_updates += 1
    learning_history.append(dict(step=learning_updates, **step_result))
    if block_step % 100 == 0:
        print(f"Training {block_step}/{training_steps} | total updates {learning_updates} | "
              f"batch loss {step_result['total']:.6f}", flush=True)

progress_result = dl.evaluate(network, training_example)
learning_trials.append(("After full live training", learning_updates, progress_result))
print("Prediction:", prediction or "Not recorded")
show_learning(dl.plot_result(training_example, progress_result, f"Trained here: {learning_updates} updates"))
learning_log(learning_trials)
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot([r["step"] for r in learning_history], [r["total"] for r in learning_history])
ax.set(xlabel="Shared-weight updates", ylabel="Training batch loss S + λR")
show_learning(fig)

### 5.5 Run inference with frozen weights

Choose a held-out pair. Its CT source slice was excluded from training, and its motion follows the same generator and range as the training pairs. Run one forward pass with frozen network weights. No optimizer step is performed. The predicted field can differ across inputs while the weights remain unchanged.

This uses the network you just trained in step 5.4. To inspect another held-out pair, change `test_slice` or `test_seed` and rerun this step.

In [ ]:
test_slice = 0  # 0, 1, 2 or 3 in the held-out slice set.
test_seed = 84
inference_network = copy.deepcopy(network).eval()
for parameter in inference_network.parameters():
    parameter.requires_grad_(False)
inference_network.zero_grad(set_to_none=True)
held_out_pair = dl.make_batch("test", seed=test_seed, index=test_slice)
frozen_before = {k: v.clone() for k, v in inference_network.state_dict().items()}
inference_result = dl.evaluate(inference_network, held_out_pair)
weights_unchanged = all(torch.equal(frozen_before[k], v) for k, v in inference_network.state_dict().items())
print("Weights unchanged:", weights_unchanged)
print("Inference field requires no optimizer updates.")
show_learning(dl.plot_result(held_out_pair, inference_result, "Held-out pair: frozen-weight inference"))

**Discuss:** trace the path from image mismatch to sampler coordinates to network weights. Which operations are needed at inference, and which are only needed for training?

References: [VoxelMorph](https://arxiv.org/abs/1809.05231), [PyTorch grid_sample](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.grid_sample.html). Dataset preparation and deformation construction are in `data/README.md`.

## 6. Q&A: discuss the registration results

Use the plots and trial logs from the practical to discuss the following.

1. **ICP and initialization:** Why can the nearest-neighbor distance decrease while the point sets remain incorrectly aligned? Why might more iterations fail to fix a poor initialization?

2. **Transformation model:** What residual mismatch remained in the translation-only CT–CT result? Why could adding rotation correct it, while further translation could not?

3. **SSD versus MI:** Why does SSD give a poor alignment in the current CT–MR experiment even though it worked for CT–CT? What assumption about corresponding intensities does SSD make, and how does MI compare the images differently?

4. **Deformable registration:** What changed when you increased the regularization weight? Use the image alignment, deformation grid and Jacobian map to explain the trade-off between fitting the images and controlling the deformation.

5. **Learning registration:** In section 4, the optimizer updates a displacement field for one image pair. What is updated during network training in section 5, and how does image mismatch provide a training signal without ground-truth displacement labels? What happens when the trained network registers a new pair?

6. **Checking the result:** What does the reported imposed-motion error measure? Why does a small value for CT–MR not establish correct anatomical alignment, and what additional evidence would you need?

### Implementation and sources

- `registration_lab.py` contains the resampler, rigid solver, metrics and regularized field objective.
- `data/README.md` records data provenance, physical geometry, preprocessing and the limits of the CT–MR reference.
- [SciPy Powell optimizer](https://docs.scipy.org/doc/scipy/reference/optimize.minimize-powell.html)
- [SimpleITK registration conventions](https://simpleitk.readthedocs.io/en/main/registrationOverview.html)
- [ITK multimodal registration example](https://itk.org/files/Examples/src/Registration/Common/PerformMultiModalityRegistrationWithMutualInformation/Documentation.html)